# Figure 1

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from plot_utils import check_data, load_integrated_table, table
check_data()


## Panel A

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge
from pyproj import CRS
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

FOLDER = Path.cwd()
if not (FOLDER / "figure1_dual_mismatch_ONE_CSV.csv").exists():
    FOLDER = Path.cwd()

ROOT = FOLDER.parents[1] if (FOLDER.parents[1] / "02_model_code").exists() else FOLDER.parent

one_csv = FOLDER / "figure1_dual_mismatch_ONE_CSV.csv"
generation_csv = str(Path.cwd() / "province_monthly_generation_profiles_source.csv")

one = load_integrated_table('figure1_dual_mismatch_ONE_CSV.csv')
spatial = one[one["source_table"].eq("diagnostic_energy_spatial_source.csv")].copy()

gen_monthly = load_integrated_table('province_monthly_generation_profiles_source.csv')

energy_cols = ["wind_twh", "solar_twh", "hydro_twh", "thermal_twh"]

annual = gen_monthly.groupby("province", as_index=False)[energy_cols].sum()
annual["total_twh"] = annual[energy_cols].sum(axis=1)
annual["green_total_twh"] = annual["wind_twh"] + annual["solar_twh"] + annual["hydro_twh"]

df = spatial[["province", "province_en", "lon", "lat"]].merge(annual, on="province", how="left")

df[energy_cols + ["total_twh", "green_total_twh"]] =\
    df[energy_cols + ["total_twh", "green_total_twh"]].fillna(0.0)

CHINA_ALBERS = CRS.from_proj4(
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=110 "
    "+datum=WGS84 +units=m +no_defs"
)

shp_files = list((Path.cwd() / 'boundaries').glob("*.shp"))
shp = next((p for p in shp_files if "bou2_4l" in p.name), None)

if shp is None:
    raise FileNotFoundError("No shapefile found")

poly = gpd.read_file(shp)

if poly.crs is None:
    poly = poly.set_crs(4326)

poly = poly.to_crs(CHINA_ALBERS)

pts = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs=4326
).to_crs(CHINA_ALBERS)

pts["x"] = pts.geometry.x
pts["y"] = pts.geometry.y

max_total = max(float(pts["total_twh"].max()), 1e-9)

pts["r_outer"] = 120000.0 * np.sqrt(pts["total_twh"] / max_total) + 60000.0

pts["r_inner"] = pts["r_outer"] * np.sqrt(
    np.clip(
        pts["green_total_twh"] / pts["total_twh"].replace(0, np.nan),
        0, 1
    )
).fillna(0.0)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Microsoft YaHei"],
    "font.size": 8
})

fig, ax = plt.subplots(figsize=(6.9, 7.5), dpi=180)

fig.patch.set_facecolor("none")
fig.patch.set_alpha(0)

ax.set_facecolor("none")
ax.patch.set_visible(False)

poly.boundary.plot(
    ax=ax,
    color="#FFFFFF00",
    linewidth=0.6,
    zorder=2
)

share_colors = {
    "wind_twh":  "#105780",
    "solar_twh": "#F4A261",
    "hydro_twh": "#3C70FF"
}

for _, row in pts.iterrows():

    x, y = row["x"], row["y"]

    ax.add_patch(Circle(
        (x, y),
        radius=row["r_outer"],
        facecolor="#AFD3FCFF",
        edgecolor="none",
        alpha=0.8,
        zorder=3
    ))

    total = row["green_total_twh"]
    if total <= 0:
        continue

    start = 90.0

    for key in ["wind_twh", "solar_twh", "hydro_twh"]:

        frac = float(row[key]) / total
        end = start - 360.0 * frac

        if frac > 0:
            ax.add_patch(Wedge(
                (x, y),
                row["r_inner"],
                end,
                start,
                facecolor=share_colors[key],
                edgecolor="white",
                linewidth=0.5,
                alpha=0.98,
                zorder=4.5
            ))

        start = end

minx, miny, maxx, maxy = poly.total_bounds

ax.set_xlim(minx - 0.04*(maxx - minx), maxx + 0.04*(maxx - minx))
ax.set_ylim(miny - 0.05*(maxy - miny), maxy + 0.05*(maxy - miny))

ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

pass

china_total = gen_monthly[energy_cols].sum()
china_green = china_total["wind_twh"] + china_total["solar_twh"] + china_total["hydro_twh"]
china_all = float(china_total.sum())

pass
pass
pass
pass
pass

axins = inset_axes(ax, width="22%", height="22%", loc="lower left", borderpad=1.2)

axins.set_xlim(0, 1)
axins.set_ylim(0, 1)
axins.set_aspect("equal")
axins.axis("off")

center = (0.5, 0.5)
outer_r = 0.38

axins.add_patch(Circle(
    center,
    outer_r,
    facecolor="#AFD3FCFF",
    edgecolor="none",
    alpha=0.8
))

inner_r = outer_r * np.sqrt(china_green / china_all) if china_all > 0 else 0

start = 90.0
for k in ["wind_twh", "solar_twh", "hydro_twh"]:
    frac = china_total[k] / china_green if china_green > 0 else 0
    end = start - 360 * frac

    if frac > 0:
        axins.add_patch(Wedge(
            center,
            inner_r,
            end,
            start,
            facecolor=share_colors[k],
            edgecolor="white",
            linewidth=0.5
        ))

    start = end

axins.text(0.5, -0.08, "China Total", ha="center", va="top", fontsize=7)

out = Path.cwd() / "figures" / "panels" / "Figure1_a.png"
out.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(
    out,
    dpi=300,
    bbox_inches="tight",
    transparent=True,
    facecolor="none",
    edgecolor="none"
)

plt.show()

pass


## Panel B

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    key = "source_table" if "source_table" in one.columns else "table"
    return one[one[key].eq(name)].copy()

PALETTE = {
    "s0": "#B04A4A", "s1": "#D7A33D", "s2": "#2F8F7B",
    "demand": "#B04A4A", "renew": "#2F8F7B",
    "solar": "#E3A51A", "wind": "#3B6EA8", "hydro": "#4DA08F",
    "grid": "#D9DEE7", "ink": "#30343B", "flex": "#7C5C99",
}

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

df = part("diagnostic_energy_spatial_source.csv")

OUT_DIR = Path.cwd()
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "panel_b_spatial_scatter.png"

BASE_FONT = 8
BASE_LINE = 0.8

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Microsoft YaHei", "SimHei", "DejaVu Sans"],
    "font.size": BASE_FONT,
    "axes.linewidth": BASE_LINE,
    "axes.edgecolor": "#30343B",
    "figure.dpi": 600,
    "savefig.dpi": 600,
})

nature_cmap = LinearSegmentedColormap.from_list(
    "nature_pastel_match",
    ["#2C5D73", "#4F8A8B", "#86B49A", "#D6C49A", "#D9A273"],
    N=256
)

fig, ax = plt.subplots(figsize=(3.15, 2.15), dpi=600)

ax.set_facecolor("white")

sc = ax.scatter(
    df["facility_demand_twh"],
    df["green_twh"],
    s=65 + df["accessible_green_twh"] * 10,
    c=df["green_demand_ratio"],
    cmap=nature_cmap,
    edgecolor="white",
    linewidth=0.28,
    alpha=0.95
)

top_pts = pd.concat([
    df.nlargest(5, "facility_demand_twh"),
    df.nlargest(5, "green_twh")
]).drop_duplicates("province")

for _, row in top_pts.iterrows():
    ax.text(
        row["facility_demand_twh"] * 1.01,
        row["green_twh"] * 1.01,
        row["province_en"],
        fontsize=BASE_FONT
    )

x_min, x_max = df["facility_demand_twh"].min(), df["facility_demand_twh"].max()
y_min, y_max = df["green_twh"].min(), df["green_twh"].max()

ax.set_xlim(x_min - (x_max - x_min) * 0.08,
            x_max + (x_max - x_min) * 0.08)

ax.set_ylim(y_min - (y_max - y_min) * 0.10,
            y_max + (y_max - y_min) * 0.10)

ax.set_xlabel("Annual facility demand (TWh)", fontsize=BASE_FONT, labelpad=4)
ax.set_ylabel("Annual wind + solar + hydro (TWh)", fontsize=BASE_FONT, labelpad=4)

ax.tick_params(
    axis="both",
    labelsize=BASE_FONT,
    pad=2,
    width=BASE_LINE,
    length=3
)

ax.grid(True, color="#D9DEE7", lw=0.55, alpha=0.78)

for spine in ax.spines.values():
    spine.set_linewidth(BASE_LINE)
    spine.set_color("#30343B")

cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.10)
cbar.set_label("Green demand ratio", fontsize=BASE_FONT, labelpad=4)
cbar.ax.tick_params(labelsize=BASE_FONT, width=BASE_LINE, pad=2)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout(pad=0.5)

plt.savefig(
    OUT_FILE,
    dpi=600,
    bbox_inches=None,
    pad_inches=0.02,
    transparent=True
)

plt.show()

pass


## Panel C

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FOLDER = Path.cwd()
CSV = FOLDER / "trial9_regional_monthly_energy_supply_horizontal_data.csv"

if not CSV.exists():
    FOLDER = Path.cwd()
    CSV = FOLDER / "trial9_regional_monthly_energy_supply_horizontal_data.csv"

df = load_integrated_table('trial9_regional_monthly_energy_supply_horizontal_data.csv')

region_order = df.sort_values("region_rank")["region"].drop_duplicates().tolist()
region_en = (
    df.sort_values("region_rank")
    .drop_duplicates("region")
    .set_index("region")["region_en"]
    .to_dict()
)

new_x = []
current_offset = 0
for i, reg in enumerate(region_order):
    sub = df[df["region"].eq(reg)].sort_values("x")

    group_x = np.arange(len(sub)) + current_offset
    new_x.extend(group_x)

    current_offset += len(sub) + 0.5

df = df.sort_values(["region_rank", "x"]).copy()
df["x"] = new_x

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Microsoft YaHei", "SimHei", "DejaVu Sans"],
    "font.size": 8,
    "axes.spines.top": False,
    "axes.linewidth": 1,
    "axes.edgecolor": "#30343B",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

fig, ax = plt.subplots(figsize=(6.9, 2.15),dpi=600)

stack_order = [
    ("Thermal", "thermal_twh", "#0AACA4"),
    ("Hydro", "hydro_twh", "#7FC6E8"),
    ("Wind", "wind_twh", "#1F78B4"),
    ("Solar", "solar_twh", "#F2D57E"),
]

bottom = np.zeros(len(df))

for label, col, color in stack_order:
    vals = df[col].to_numpy()
    ax.bar(
        df["x"],
        vals,
        bottom=bottom,
        width=0.74,
        color=color,
        edgecolor="white",
        linewidth=0.28,
        label=label,
        zorder=3,
    )
    bottom += vals

gap = 0.6
centers = []
boundaries = []

for reg in region_order:
    sub = df[df["region"].eq(reg)]
    centers.append((sub["x"].min() + sub["x"].max()) / 2)
    boundaries.append(sub["x"].max() + gap / 2)

for b in boundaries[:-1]:
    ax.axvline(b, color="#6B7280", lw=0.85, alpha=0.72, zorder=3.5)

ax.set_xticks(centers, [region_en[r] for r in region_order])
ax.tick_params(axis="x", length=0, pad=8)

ax.set_xlim(df["x"].min() - 0.4, df["x"].max() + 0.4)

ax.set_ylim(0, bottom.max() * 1.02)

ax.set_ylabel("Regional power generation (TWh)")
ax.grid(True, axis="y", color="#D9DEE7", lw=0.55, alpha=0.78, zorder=0)

for spine in ax.spines.values():
    spine.set_zorder(5)
ax.tick_params(axis="both", zorder=5)

ax.legend(
    fontsize=7,
    frameon=False,
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),
)

plt.tight_layout(pad=0.6)

out = FOLDER / "trial9_regional_monthly_energy_supply_compact.png"
fig.savefig(out, dpi=260, bbox_inches="tight", pad_inches=0.02)

plt.show()

pass


## Panel D

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import Image, display

mpl.rcdefaults()
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.spines.top':False,'axes.spines.right':False})

SOLAR = '#E6AA36'
TEAL = '#2D9C82'
RED = '#C95B4A'
WIND = '#477DB1'
GRID = '#DCE4EA'

FONT_SIZE = 8
TICK_SIZE = 8
LABEL_SIZE = 8
LEFT_LABEL_SIZE = 7
LEGEND_SIZE = 7.2

def part(data: pd.DataFrame, name: str) -> pd.DataFrame:
    return data[data["atlas_table"].eq(name)].copy().dropna(axis=1, how="all")

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "main_figure1_d.png"

data = pd.read_feather(DATA)
df = part(data, "f1_monthly_national").sort_values("month")

fig, ax = plt.subplots(figsize=(4.6, 1.9), dpi=300)

ax.plot(
    df["month"],
    df["facility_demand_twh"],
    color=RED,
    lw=2.1,
    marker="o",
    label="Facility demand"
)

ax.plot(
    df["month"],
    df["accessible_green_twh"],
    color=TEAL,
    lw=2.1,
    marker="o",
    label="Accessible clean"
)

ax.fill_between(
    df["month"],
    df["accessible_green_twh"],
    df["facility_demand_twh"],
    where=df["facility_demand_twh"] > df["accessible_green_twh"],
    color=RED,
    alpha=0.1
)

ax.set_ylabel("Monthly electricity (TWh)", fontsize=LEFT_LABEL_SIZE)

ax.tick_params(axis="both", labelsize=TICK_SIZE)

ax.grid(
    True,
    color=GRID,
    lw=0.55,
    alpha=0.55
)

ax2 = ax.twinx()

ax2.plot(
    df["month"],
    df["solar_index"],
    color=SOLAR,
    lw=1.5,
    ls="--",
    label="Solar"
)

ax2.plot(
    df["month"],
    df["wind_index"],
    color=WIND,
    lw=1.5,
    ls="--",
    label="Wind"
)

ax2.set_ylabel("Generation index", fontsize=LABEL_SIZE)
ax2.tick_params(axis="y", labelsize=TICK_SIZE)

for _a in (ax, ax2):
    _a.spines["top"].set_visible(False)
    _a.spines["right"].set_visible(False)

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()

ax.legend(
    h1 + h2,
    l1 + l2,
    fontsize=LEGEND_SIZE,
    loc="upper left",
    frameon=True
)

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=180,
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor="white",
    edgecolor="none"
)

plt.show()

pass


## Panel E

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INTEGRATED_DATA = Path.cwd() / "data/current_results_legacy_plot_schema.feather"

def load_integrated_table(source_csv, **kwargs):
    bundle = pd.read_feather(INTEGRATED_DATA)
    rows = bundle[bundle["source_csv"].eq(source_csv)].copy()

    if rows.empty:
        raise FileNotFoundError(f"No integrated rows found for {source_csv}")

    prefix = rows["source_prefix"].iloc[0]
    dtype_map = json.loads(rows["source_dtypes_json"].iloc[0])

    columns = [c for c in rows.columns if c.startswith(prefix)]
    result = rows[columns].copy()
    result.columns = [c[len(prefix):] for c in columns]

    for column, dtype in dtype_map.items():
        if column not in result.columns:
            continue

        if dtype.startswith("int"):
            result[column] = pd.to_numeric(result[column], errors="raise").astype(dtype)
        elif dtype.startswith("float"):
            result[column] = pd.to_numeric(result[column], errors="coerce").astype(dtype)
        elif dtype == "bool":
            result[column] = result[column].astype(bool)
        elif dtype == "object":
            result[column] = result[column].where(result[column].notna(), None).astype(object)

    return result.reset_index(drop=True)

GRID = "#DCE4EA"

FONT_SIZE = 8
TICK_SIZE = 8
LABEL_SIZE = 8
LEGEND_SIZE = 7.2

AXIS_LW = 0.8
TICK_LW = 0.8

PUE_COLOR = "#7C4D79"
WUE_COLOR = "#2E5E9E"

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

key = "source_table" if "source_table" in one.columns else "table"
df = one[one[key].eq("figure1_enriched_daily_profile_source.csv")].copy()
df = df.sort_values("day_of_year")

fig, ax = plt.subplots(figsize=(4.6, 1.3), dpi=300)

pue_line, = ax.plot(
    df["day_of_year"],
    df["weighted_pue"],
    color=PUE_COLOR,
    lw=1.8,
    alpha=1.0,
    ls="-",
    label="Weighted PUE",
    zorder=10
)

ax.set_xlabel("Month", fontsize=LABEL_SIZE)
ax.set_ylabel("PUE", fontsize=LABEL_SIZE)

ax.set_xticks(
    [15, 74, 135, 196, 258, 319],
    ["Jan", "Mar", "May", "Jul", "Sep", "Nov"]
)

ax.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
    width=TICK_LW,
    length=3
)

ax.grid(
    True,
    color=GRID,
    lw=0.55,
    alpha=0.55,
    zorder=0
)

ax2 = ax.twinx()

ax.set_zorder(2)
ax2.set_zorder(1)
ax.patch.set_visible(False)

wue_line, = ax2.plot(
    df["day_of_year"],
    df["weighted_wue_l_per_kwh_it"],
    color=WUE_COLOR,
    lw=1.8,
    ls="-",
    alpha=0.55,
    label="Weighted WUE",
    zorder=3
)

ax2.set_ylabel("WUE (L kWh$^{-1}$ IT)", fontsize=LABEL_SIZE)

ax2.tick_params(
    axis="y",
    labelsize=TICK_SIZE,
    width=TICK_LW,
    length=3
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(AXIS_LW)
ax.spines["bottom"].set_linewidth(AXIS_LW)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax2.spines["bottom"].set_visible(False)

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()

ax.legend(
    h1 + h2,
    l1 + l2,
    fontsize=LEGEND_SIZE,
    loc="upper left",
    frameon=True
)

OUT = Path.cwd() / "figures" / "panels" / "main_figure1_pue_wue.png"
OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=180,
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor="white",
    edgecolor="none"
)

plt.show()

pass


## Panel F

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    key = "source_table" if "source_table" in one.columns else "table"
    return one[one[key].eq(name)].copy()

PALETTE = {
    "s0": "#B04A4A",
    "s1": "#D7A33D",
    "s2": "#2F8F7B",
    "demand": "#B04A4A",
    "renew": "#2F8F7B",
    "solar": "#E3A51A",
    "wind": "#3B6EA8",
    "hydro": "#4DA08F",
    "grid": "#D9DEE7",
    "ink": "#30343B",
    "flex": "#7C5C99",
}

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    _key = "source_table" if "source_table" in one.columns else "table"
    return one[one[_key].eq(name)].copy()

TICK_SIZE = 8.5
LABEL_SIZE = 9.0
TITLE_SIZE = 9.0
AXIS_LW = 0.8

df = part("figure1_enriched_seasonal_24h_source.csv")
sub = df[df["season"].eq("Spring")].copy()

fig, ax = plt.subplots(figsize=(2.3, 1.8), dpi=180)

ax.axvspan(0, 7, color="#DCE9F7", alpha=0.65, lw=0)
ax.axvspan(7, 16, color="#F8E9B8", alpha=0.60, lw=0)
ax.axvspan(16, 24, color="#F7D8D6", alpha=0.60, lw=0)

ax.plot(
    sub["hour"],
    sub["demand_norm"],
    color=PALETTE["s0"],
    lw=2.0,
    label="Demand"
)

ax.plot(
    sub["hour"],
    sub["solar_norm"],
    color="#E5A11F",
    lw=1.7,
    label="Solar"
)

ax.plot(
    sub["hour"],
    sub["wind_norm"],
    color="#4B79B7",
    lw=1.7,
    label="Wind"
)

ax.plot(
    sub["hour"],
    sub["hydro_norm"],
    color="#2D8C76",
    lw=1.5,
    ls="--",
    label="Hydro"
)

ax.plot(
    sub["hour"],
    sub["accessible_green_norm"],
    color=PALETTE["renew"],
    lw=1.6,
    ls="-.",
    label="Accessible green"
)

ax.set_xlim(0, 23)
ax.set_ylim(0, 1.15)
ax.set_xticks([0, 6, 12, 18, 23])

ax.set_xlabel("Hour", fontsize=LABEL_SIZE)
ax.set_ylabel("Normalized value", fontsize=LABEL_SIZE)

ax.set_title(
    'h',
    loc="left",
    fontweight="bold",
    fontsize=TITLE_SIZE
)

ax.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
    width=AXIS_LW,
    length=3
)

ax.grid(
    True,
    color=PALETTE["grid"],
    lw=0.45,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(AXIS_LW)
ax.spines["bottom"].set_linewidth(AXIS_LW)

ax.legend(fontsize=5.9, loc="upper left", ncol=1, frameon=True)

plt.show()


## Panel F

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    key = "source_table" if "source_table" in one.columns else "table"
    return one[one[key].eq(name)].copy()

PALETTE = {
    "s0": "#B04A4A",
    "s1": "#D7A33D",
    "s2": "#2F8F7B",
    "demand": "#B04A4A",
    "renew": "#2F8F7B",
    "solar": "#E3A51A",
    "wind": "#3B6EA8",
    "hydro": "#4DA08F",
    "grid": "#D9DEE7",
    "ink": "#30343B",
    "flex": "#7C5C99",
}

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    _key = "source_table" if "source_table" in one.columns else "table"
    return one[one[_key].eq(name)].copy()

TICK_SIZE = 8.5
LABEL_SIZE = 9.0
TITLE_SIZE = 9.0
AXIS_LW = 0.8

df = part("figure1_enriched_seasonal_24h_source.csv")
sub = df[df["season"].eq("Summer")].copy()

fig, ax = plt.subplots(figsize=(2.3, 1.8), dpi=180)

ax.axvspan(0, 7, color="#DCE9F7", alpha=0.65, lw=0)
ax.axvspan(7, 16, color="#F8E9B8", alpha=0.60, lw=0)
ax.axvspan(16, 24, color="#F7D8D6", alpha=0.60, lw=0)

ax.plot(
    sub["hour"],
    sub["demand_norm"],
    color=PALETTE["s0"],
    lw=2.0,
    label="Demand"
)

ax.plot(
    sub["hour"],
    sub["solar_norm"],
    color="#E5A11F",
    lw=1.7,
    label="Solar"
)

ax.plot(
    sub["hour"],
    sub["wind_norm"],
    color="#4B79B7",
    lw=1.7,
    label="Wind"
)

ax.plot(
    sub["hour"],
    sub["hydro_norm"],
    color="#2D8C76",
    lw=1.5,
    ls="--",
    label="Hydro"
)

ax.plot(
    sub["hour"],
    sub["accessible_green_norm"],
    color=PALETTE["renew"],
    lw=1.6,
    ls="-.",
    label="Accessible green"
)

ax.set_xlim(0, 23)
ax.set_ylim(0, 1.15)
ax.set_xticks([0, 6, 12, 18, 23])

ax.set_xlabel("Hour", fontsize=LABEL_SIZE)
ax.set_ylabel("Normalized value", fontsize=LABEL_SIZE)

ax.set_title(
    'i',
    loc="left",
    fontweight="bold",
    fontsize=TITLE_SIZE
)

ax.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
    width=AXIS_LW,
    length=3
)

ax.grid(
    True,
    color=PALETTE["grid"],
    lw=0.45,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(AXIS_LW)
ax.spines["bottom"].set_linewidth(AXIS_LW)

plt.show()


## Panel F

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    key = "source_table" if "source_table" in one.columns else "table"
    return one[one[key].eq(name)].copy()

PALETTE = {
    "s0": "#B04A4A",
    "s1": "#D7A33D",
    "s2": "#2F8F7B",
    "demand": "#B04A4A",
    "renew": "#2F8F7B",
    "solar": "#E3A51A",
    "wind": "#3B6EA8",
    "hydro": "#4DA08F",
    "grid": "#D9DEE7",
    "ink": "#30343B",
    "flex": "#7C5C99",
}

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    _key = "source_table" if "source_table" in one.columns else "table"
    return one[one[_key].eq(name)].copy()

TICK_SIZE = 8.5
LABEL_SIZE = 9.0
TITLE_SIZE = 9.0
AXIS_LW = 0.8

df = part("figure1_enriched_seasonal_24h_source.csv")
sub = df[df["season"].eq("Autumn")].copy()

fig, ax = plt.subplots(figsize=(2.3, 1.8), dpi=180)

ax.axvspan(0, 7, color="#DCE9F7", alpha=0.65, lw=0)
ax.axvspan(7, 16, color="#F8E9B8", alpha=0.60, lw=0)
ax.axvspan(16, 24, color="#F7D8D6", alpha=0.60, lw=0)

ax.plot(
    sub["hour"],
    sub["demand_norm"],
    color=PALETTE["s0"],
    lw=2.0,
    label="Demand"
)

ax.plot(
    sub["hour"],
    sub["solar_norm"],
    color="#E5A11F",
    lw=1.7,
    label="Solar"
)

ax.plot(
    sub["hour"],
    sub["wind_norm"],
    color="#4B79B7",
    lw=1.7,
    label="Wind"
)

ax.plot(
    sub["hour"],
    sub["hydro_norm"],
    color="#2D8C76",
    lw=1.5,
    ls="--",
    label="Hydro"
)

ax.plot(
    sub["hour"],
    sub["accessible_green_norm"],
    color=PALETTE["renew"],
    lw=1.6,
    ls="-.",
    label="Accessible green"
)

ax.set_xlim(0, 23)
ax.set_ylim(0, 1.15)
ax.set_xticks([0, 6, 12, 18, 23])

ax.set_xlabel("Hour", fontsize=LABEL_SIZE)
ax.set_ylabel("Normalized value", fontsize=LABEL_SIZE)

ax.set_title(
    'j',
    loc="left",
    fontweight="bold",
    fontsize=TITLE_SIZE
)

ax.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
    width=AXIS_LW,
    length=3
)

ax.grid(
    True,
    color=PALETTE["grid"],
    lw=0.45,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(AXIS_LW)
ax.spines["bottom"].set_linewidth(AXIS_LW)

plt.show()


## Panel F

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    key = "source_table" if "source_table" in one.columns else "table"
    return one[one[key].eq(name)].copy()

PALETTE = {
    "s0": "#B04A4A",
    "s1": "#D7A33D",
    "s2": "#2F8F7B",
    "demand": "#B04A4A",
    "renew": "#2F8F7B",
    "solar": "#E3A51A",
    "wind": "#3B6EA8",
    "hydro": "#4DA08F",
    "grid": "#D9DEE7",
    "ink": "#30343B",
    "flex": "#7C5C99",
}

one = load_integrated_table("figure1_dual_mismatch_ONE_CSV.csv")

def part(name):
    _key = "source_table" if "source_table" in one.columns else "table"
    return one[one[_key].eq(name)].copy()

TICK_SIZE = 8.5
LABEL_SIZE = 9.0
TITLE_SIZE = 9.0
AXIS_LW = 0.8

df = part("figure1_enriched_seasonal_24h_source.csv")
sub = df[df["season"].eq("Winter")].copy()

fig, ax = plt.subplots(figsize=(2.3, 1.8), dpi=180)

ax.axvspan(0, 7, color="#DCE9F7", alpha=0.65, lw=0)
ax.axvspan(7, 16, color="#F8E9B8", alpha=0.60, lw=0)
ax.axvspan(16, 24, color="#F7D8D6", alpha=0.60, lw=0)

ax.plot(
    sub["hour"],
    sub["demand_norm"],
    color=PALETTE["s0"],
    lw=2.0,
    label="Demand"
)

ax.plot(
    sub["hour"],
    sub["solar_norm"],
    color="#E5A11F",
    lw=1.7,
    label="Solar"
)

ax.plot(
    sub["hour"],
    sub["wind_norm"],
    color="#4B79B7",
    lw=1.7,
    label="Wind"
)

ax.plot(
    sub["hour"],
    sub["hydro_norm"],
    color="#2D8C76",
    lw=1.5,
    ls="--",
    label="Hydro"
)

ax.plot(
    sub["hour"],
    sub["accessible_green_norm"],
    color=PALETTE["renew"],
    lw=1.6,
    ls="-.",
    label="Accessible green"
)

ax.set_xlim(0, 23)
ax.set_ylim(0, 1.15)
ax.set_xticks([0, 6, 12, 18, 23])

ax.set_xlabel("Hour", fontsize=LABEL_SIZE)
ax.set_ylabel("Normalized value", fontsize=LABEL_SIZE)

ax.set_title(
    'k',
    loc="left",
    fontweight="bold",
    fontsize=TITLE_SIZE
)

ax.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
    width=AXIS_LW,
    length=3
)

ax.grid(
    True,
    color=PALETTE["grid"],
    lw=0.45,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(AXIS_LW)
ax.spines["bottom"].set_linewidth(AXIS_LW)

plt.show()


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_renewable_stack_monthly.png"

data = pd.read_feather(
    DATA,
    columns=["atlas_table", "profile_variable", "province_en", "month", "value"],
)

data = data[
    data["atlas_table"].eq("f1_supp_daily_province_profiles")
    & data["profile_variable"].isin(["wind", "solar", "hydro"])
].copy()

data = (
    data.pivot_table(
        index=["province_en", "month"],
        columns="profile_variable",
        values="value",
        aggfunc="sum",
    )
    .reset_index()
    .fillna(0.0)
)

provinces = sorted(data["province_en"].dropna().unique())

colors = {
    "wind":  "#3B6EA8",
    "solar": "#D7A33D",
    "hydro": "#2F8F7B",
}

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.8

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):

    panel_data = data[data["province_en"].eq(province)].sort_values("month")

    month = panel_data["month"].to_numpy(dtype=float)
    wind = panel_data["wind"].to_numpy(dtype=float)
    solar = panel_data["solar"].to_numpy(dtype=float)
    hydro = panel_data["hydro"].to_numpy(dtype=float)

    ax.stackplot(
        month,
        wind,
        solar,
        hydro,
        colors=[
            colors["wind"],
            colors["solar"],
            colors["hydro"],
        ],
        alpha=0.84,
        linewidth=0.55,
        edgecolor="white",
    )

    ax.plot(
        month,
        wind,
        color="white",
        lw=0.55,
        alpha=0.95,
        zorder=4
    )

    ax.plot(
        month,
        wind + solar,
        color="white",
        lw=0.55,
        alpha=0.95,
        zorder=4
    )

    ax.plot(
        month,
        wind + solar + hydro,
        color="white",
        lw=0.45,
        alpha=0.75,
        zorder=4
    )

    ax.text(
        0.025,
        0.955,
        province,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(1, 12)
    ax.set_ylim(bottom=0)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([1, 4, 7, 10, 12])
        ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct", "Dec"])

handles = [
    Patch(
        facecolor=colors["wind"],
        edgecolor="none",
        alpha=0.84,
        label="Wind"
    ),
    Patch(
        facecolor=colors["solar"],
        edgecolor="none",
        alpha=0.84,
        label="Solar"
    ),
    Patch(
        facecolor=colors["hydro"],
        edgecolor="none",
        alpha=0.84,
        label="Hydro"
    ),
]

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=3,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.7,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=3,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_pue_daily_deciles.png"

data = pd.read_feather(
    DATA,
    columns=[
        "atlas_table",
        "profile_variable",
        "province_en",
        "day_of_year",
        "q10",
        "q50",
        "q90",
    ],
)

data = data[
    data["atlas_table"].eq("f1_supp_daily_province_profiles")
    & data["profile_variable"].eq("pue")
].copy()

provinces = sorted(data["province_en"].dropna().unique())

PUE_COLOR = "#7A6AA6"

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.6

Q10_LW = 0.75
Q50_LW = 1.35
Q90_LW = 0.75

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):

    panel_data = data[data["province_en"].eq(province)].sort_values("day_of_year")

    x = panel_data["day_of_year"].to_numpy(dtype=float)
    q10 = panel_data["q10"].to_numpy(dtype=float)
    q50 = panel_data["q50"].to_numpy(dtype=float)
    q90 = panel_data["q90"].to_numpy(dtype=float)

    ax.fill_between(
        x,
        q10,
        q90,
        color=PUE_COLOR,
        alpha=0.16,
        linewidth=0,
        zorder=1
    )

    ax.plot(
        x,
        q10,
        color=PUE_COLOR,
        lw=Q10_LW,
        alpha=0.72,
        ls="--",
        zorder=3
    )

    ax.plot(
        x,
        q50,
        color=PUE_COLOR,
        lw=Q50_LW,
        alpha=1.0,
        zorder=4
    )

    ax.plot(
        x,
        q90,
        color=PUE_COLOR,
        lw=Q90_LW,
        alpha=0.72,
        ls=":",
        zorder=3
    )

    ax.text(
        0.025,
        0.955,
        province,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(1, 365)

    ax.grid(False)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([1, 91, 182, 274, 365])
        ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct", "Dec"])

legend_ax = None

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

handles = [
    Line2D(
        [0],
        [0],
        color=PUE_COLOR,
        lw=Q10_LW,
        alpha=0.72,
        ls="--",
        label="Q10"
    ),
    Line2D(
        [0],
        [0],
        color=PUE_COLOR,
        lw=Q50_LW,
        label="Q50"
    ),
    Line2D(
        [0],
        [0],
        color=PUE_COLOR,
        lw=Q90_LW,
        alpha=0.72,
        ls=":",
        label="Q90"
    ),
    Patch(
        facecolor=PUE_COLOR,
        alpha=0.16,
        edgecolor="none",
        label="Q10-Q90"
    ),
]

if legend_ax is not None:
    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.8,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )
else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white"
)

plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_wue_daily_deciles.png"

data = pd.read_feather(
    DATA,
    columns=[
        "atlas_table",
        "profile_variable",
        "province_en",
        "day_of_year",
        "q10",
        "q50",
        "q90",
    ],
)

data = data[
    data["atlas_table"].eq("f1_supp_daily_province_profiles")
    & data["profile_variable"].eq("wue")
].copy()

provinces = sorted(data["province_en"].dropna().unique())

WUE_COLOR = "#4F8D74"

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.6

Q10_LW = 0.75
Q50_LW = 1.35
Q90_LW = 0.75

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):

    panel_data = data[data["province_en"].eq(province)].sort_values("day_of_year")

    x = panel_data["day_of_year"].to_numpy(dtype=float)
    q10 = panel_data["q10"].to_numpy(dtype=float)
    q50 = panel_data["q50"].to_numpy(dtype=float)
    q90 = panel_data["q90"].to_numpy(dtype=float)

    ax.fill_between(
        x,
        np.minimum(q10, q90),
        np.maximum(q10, q90),
        color=WUE_COLOR,
        alpha=0.16,
        linewidth=0,
        zorder=1
    )

    ax.plot(
        x,
        q10,
        color=WUE_COLOR,
        lw=Q10_LW,
        alpha=0.72,
        ls="--",
        zorder=3
    )

    ax.plot(
        x,
        q50,
        color=WUE_COLOR,
        lw=Q50_LW,
        alpha=1.0,
        zorder=4
    )

    ax.plot(
        x,
        q90,
        color=WUE_COLOR,
        lw=Q90_LW,
        alpha=0.72,
        ls=":",
        zorder=3
    )

    ax.text(
        0.025,
        0.955,
        province,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(1, 365)

    ax.grid(False)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([1, 91, 182, 274, 365])
        ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct", "Dec"])

legend_ax = None

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

handles = [
    Line2D(
        [0],
        [0],
        color=WUE_COLOR,
        lw=Q10_LW,
        alpha=0.72,
        ls="--",
        label="PUE-Q10"
    ),
    Line2D(
        [0],
        [0],
        color=WUE_COLOR,
        lw=Q50_LW,
        label="PUE-Q50"
    ),
    Line2D(
        [0],
        [0],
        color=WUE_COLOR,
        lw=Q90_LW,
        alpha=0.72,
        ls=":",
        label="PUE-Q90"
    ),
    Patch(
        facecolor=WUE_COLOR,
        alpha=0.16,
        edgecolor="none",
        label="PUE-Q10-Q90"
    ),
]

if legend_ax is not None:
    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.8,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )
else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white"
)

plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_spring_province_hourly.png"

data = pd.read_feather(
    DATA,
    columns=[
        "atlas_table", "season", "province_en", "hour",
        "demand_norm", "solar_norm", "wind_norm", "hydro_norm",
        "accessible_green_norm",
    ],
)

data = data[
    data["atlas_table"].eq("f1_supp_province_season_hourly")
    & data["season"].eq("Spring")
].copy()

provinces = sorted(data["province_en"].dropna().unique())

colors = {
    "demand": "#B04A4A",
    "solar": "#E3A51A",
    "wind": "#4C78A8",
    "hydro": "#2F8F7B",
    "green": "#238B73",
}

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.6

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True,
    sharey=True,
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):
    panel_data = data[data["province_en"].eq(province)].sort_values("hour")
    hour = panel_data["hour"].to_numpy(dtype=float)

    ax.axvspan(0, 7, color="#DCE9F7", alpha=0.42, lw=0)
    ax.axvspan(7, 16, color="#F8E9B8", alpha=0.38, lw=0)
    ax.axvspan(16, 23, color="#F7D8D6", alpha=0.38, lw=0)

    ax.plot(hour, panel_data["demand_norm"], color=colors["demand"], lw=1.35, zorder=4)
    ax.plot(hour, panel_data["solar_norm"], color=colors["solar"], lw=1.15, zorder=4)
    ax.plot(hour, panel_data["wind_norm"], color=colors["wind"], lw=1.15, zorder=4)
    ax.plot(hour, panel_data["hydro_norm"], color=colors["hydro"], lw=1.0, ls="--", zorder=4)
    ax.plot(hour, panel_data["accessible_green_norm"], color=colors["green"], lw=1.05, ls="-.", zorder=4)

    ax.text(
        0.025, 0.955, province,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(0, 23)
    ax.set_ylim(0, 1.12)

    ax.grid(False)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([0, 6, 12, 18, 23])

legend_ax = None

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

handles = [
    Line2D([0], [0], color=colors["demand"], lw=1.5, label="Demand"),
    Line2D([0], [0], color=colors["solar"], lw=1.3, label="Solar"),
    Line2D([0], [0], color=colors["wind"], lw=1.3, label="Wind"),
    Line2D([0], [0], color=colors["green"], lw=1.2, ls="-.", label="Accessible green"),
]

if legend_ax is not None:
    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.8,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )
else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white"
)

plt.show()


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_summer_province_hourly.png"

data = pd.read_feather(
    DATA,
    columns=[
        "atlas_table", "season", "province_en", "hour",
        "demand_norm", "solar_norm", "wind_norm", "hydro_norm",
        "accessible_green_norm",
    ],
)

data = data[
    data["atlas_table"].eq("f1_supp_province_season_hourly")
    & data["season"].eq("Summer")
].copy()

provinces = sorted(data["province_en"].dropna().unique())

colors = {
    "demand": "#B04A4A",
    "solar": "#E3A51A",
    "wind": "#4C78A8",
    "hydro": "#2F8F7B",
    "green": "#238B73",
}

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.6

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True,
    sharey=True,
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):
    panel_data = data[data["province_en"].eq(province)].sort_values("hour")
    hour = panel_data["hour"].to_numpy(dtype=float)

    ax.axvspan(0, 7, color="#DCE9F7", alpha=0.42, lw=0)
    ax.axvspan(7, 16, color="#F8E9B8", alpha=0.38, lw=0)
    ax.axvspan(16, 23, color="#F7D8D6", alpha=0.38, lw=0)

    ax.plot(hour, panel_data["demand_norm"], color=colors["demand"], lw=1.35, zorder=4)
    ax.plot(hour, panel_data["solar_norm"], color=colors["solar"], lw=1.15, zorder=4)
    ax.plot(hour, panel_data["wind_norm"], color=colors["wind"], lw=1.15, zorder=4)
    ax.plot(hour, panel_data["hydro_norm"], color=colors["hydro"], lw=1.0, ls="--", zorder=4)
    ax.plot(hour, panel_data["accessible_green_norm"], color=colors["green"], lw=1.05, ls="-.", zorder=4)

    ax.text(
        0.025, 0.955, province,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(0, 23)
    ax.set_ylim(0, 1.12)

    ax.grid(False)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([0, 6, 12, 18, 23])

legend_ax = None

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

handles = [
    Line2D([0], [0], color=colors["demand"], lw=1.5, label="Demand"),
    Line2D([0], [0], color=colors["solar"], lw=1.3, label="Solar"),
    Line2D([0], [0], color=colors["wind"], lw=1.3, label="Wind"),
    Line2D([0], [0], color=colors["green"], lw=1.2, ls="-.", label="Accessible green"),
]

if legend_ax is not None:
    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.8,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )
else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white"
)

plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_autumn_province_hourly.png"

data = pd.read_feather(
    DATA,
    columns=[
        "atlas_table", "season", "province_en", "hour",
        "demand_norm", "solar_norm", "wind_norm", "hydro_norm",
        "accessible_green_norm",
    ],
)

data = data[
    data["atlas_table"].eq("f1_supp_province_season_hourly")
    & data["season"].eq("Autumn")
].copy()

provinces = sorted(data["province_en"].dropna().unique())

colors = {
    "demand": "#B04A4A",
    "solar": "#E3A51A",
    "wind": "#4C78A8",
    "hydro": "#2F8F7B",
    "green": "#238B73",
}

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.6

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True,
    sharey=True,
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):
    panel_data = data[data["province_en"].eq(province)].sort_values("hour")
    hour = panel_data["hour"].to_numpy(dtype=float)

    ax.axvspan(0, 7, color="#DCE9F7", alpha=0.42, lw=0)
    ax.axvspan(7, 16, color="#F8E9B8", alpha=0.38, lw=0)
    ax.axvspan(16, 23, color="#F7D8D6", alpha=0.38, lw=0)

    ax.plot(
        hour,
        panel_data["demand_norm"],
        color=colors["demand"],
        lw=1.35,
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["solar_norm"],
        color=colors["solar"],
        lw=1.15,
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["wind_norm"],
        color=colors["wind"],
        lw=1.15,
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["hydro_norm"],
        color=colors["hydro"],
        lw=1.0,
        ls="--",
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["accessible_green_norm"],
        color=colors["green"],
        lw=1.05,
        ls="-.",
        zorder=4
    )

    ax.text(
        0.025,
        0.955,
        province,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(0, 23)
    ax.set_ylim(0, 1.12)

    ax.grid(False)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([0, 6, 12, 18, 23])

legend_ax = None

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

handles = [
    Line2D(
        [0],
        [0],
        color=colors["demand"],
        lw=1.5,
        label="Demand"
    ),
    Line2D(
        [0],
        [0],
        color=colors["solar"],
        lw=1.3,
        label="Solar"
    ),
    Line2D(
        [0],
        [0],
        color=colors["wind"],
        lw=1.3,
        label="Wind"
    ),
    Line2D(
        [0],
        [0],
        color=colors["green"],
        lw=1.2,
        ls="-.",
        label="Accessible green"
    ),
]

if legend_ax is not None:
    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.8,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )
else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white"
)

plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "figure1_supp_candidate_winter_province_hourly.png"

data = pd.read_feather(
    DATA,
    columns=[
        "atlas_table", "season", "province_en", "hour",
        "demand_norm", "solar_norm", "wind_norm", "hydro_norm",
        "accessible_green_norm",
    ],
)

data = data[
    data["atlas_table"].eq("f1_supp_province_season_hourly")
    & data["season"].eq("Winter")
].copy()

provinces = sorted(data["province_en"].dropna().unique())

colors = {
    "demand": "#B04A4A",
    "solar": "#E3A51A",
    "wind": "#4C78A8",
    "hydro": "#2F8F7B",
    "green": "#238B73",
}

SPINE_LW = 1.15
TICK_LW = 0.9

BASE_FONT = 8.2
TICK_SIZE = 7.0
PROVINCE_FONT = 8.4
LEGEND_FONT = 8.6

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": BASE_FONT,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": SPINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
})

ncols = 4
nrows = int(np.ceil(len(provinces) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10.6, 15.2),
    sharex=True,
    sharey=True,
)

axes = np.asarray(axes).ravel()

for ax, province in zip(axes, provinces):
    panel_data = data[data["province_en"].eq(province)].sort_values("hour")
    hour = panel_data["hour"].to_numpy(dtype=float)

    ax.axvspan(0, 7, color="#DCE9F7", alpha=0.42, lw=0)
    ax.axvspan(7, 16, color="#F8E9B8", alpha=0.38, lw=0)
    ax.axvspan(16, 23, color="#F7D8D6", alpha=0.38, lw=0)

    ax.plot(
        hour,
        panel_data["demand_norm"],
        color=colors["demand"],
        lw=1.35,
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["solar_norm"],
        color=colors["solar"],
        lw=1.15,
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["wind_norm"],
        color=colors["wind"],
        lw=1.15,
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["hydro_norm"],
        color=colors["hydro"],
        lw=1.0,
        ls="--",
        zorder=4
    )

    ax.plot(
        hour,
        panel_data["accessible_green_norm"],
        color=colors["green"],
        lw=1.05,
        ls="-.",
        zorder=4
    )

    ax.text(
        0.025,
        0.955,
        province,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PROVINCE_FONT,
        fontweight="bold",
        color="#30343B",
    )

    ax.set_xlim(0, 23)
    ax.set_ylim(0, 1.12)

    ax.grid(False)

    ax.tick_params(
        labelsize=TICK_SIZE,
        length=2.6,
        pad=1.8,
        width=TICK_LW
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#5F6873")
        spine.set_linewidth(SPINE_LW)

bottom_start = (nrows - 1) * ncols

for i, ax in enumerate(axes[:len(provinces)]):
    if i >= bottom_start:
        ax.set_xticks([0, 6, 12, 18, 23])

legend_ax = None

if len(provinces) < len(axes):
    legend_ax = axes[len(provinces)]
    legend_ax.set_visible(True)
    legend_ax.axis("off")

    for ax in axes[len(provinces) + 1:]:
        ax.set_visible(False)

fig.tight_layout(
    rect=(0.015, 0.015, 0.995, 0.995),
    h_pad=1.0,
    w_pad=0.8
)

handles = [
    Line2D(
        [0],
        [0],
        color=colors["demand"],
        lw=1.5,
        label="Demand"
    ),
    Line2D(
        [0],
        [0],
        color=colors["solar"],
        lw=1.3,
        label="Solar"
    ),
    Line2D(
        [0],
        [0],
        color=colors["wind"],
        lw=1.3,
        label="Wind"
    ),
    Line2D(
        [0],
        [0],
        color=colors["green"],
        lw=1.2,
        ls="-.",
        label="Accessible green"
    ),
]

if legend_ax is not None:
    legend_ax.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
        handlelength=1.8,
        columnspacing=1.25,
        handletextpad=0.45,
        borderaxespad=0.0,
    )
else:
    fig.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(0.985, 0.02),
        ncol=2,
        frameon=False,
        fontsize=LEGEND_FONT,
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white"
)

plt.show()

pass
